# Directed Graphs
Now our edges (*x*,*y*) **are ordered pairs**, so (*x*,*y*) is an edge **from** *x* **to** *y*, and **is not** the same as (*y*,*x*). 

Both BFS and DFS have natural analogues, with the resulting Trees expressing slightly different relationships between the starting node s and every node in its corresponding *T*. 

In general, and most crucially, for some node *s* in BFS(*s*) or DFS(*s*). a node *x* in the resulted connected component *T* still implies path from *s* to *x*, but not necessarily a path from *x* **to** *s*.

In [15]:
from dataclasses import dataclass
from typing import TypedDict
from __future__ import annotations

e = [(1,2),(1,3),(2,4),(2,5),(2,6),(3,5),(3,6),(3,7),(4,8),(4,9),(5,10),(5,11),
(6,11),(6,12),(7,13),(7,14),(2,1),(4,2),(3,1),(7,3),(5,2),(6,3),(8,1),(11,1),(15,1)]
n = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15]


class LinkedList:
    class Node:
        def __init__(self, data):
            self.data: int = data
            self.next: Node= None

    def __init__(self):
        self.head: Node = None  # The entry point of the list
        self.tail: Node = None
    # Add a node at the end of the list
    def append(self, data: int):
        new_node = self.Node(data)
        if not self.head:
            self.head = new_node
            self.tail = new_node
            return
        current = self.head
        while current.next:  # Traverse to the last node
            current = current.next
        current.next = new_node
        self.tail = new_node
    
    def __contains__(self, data: int) -> bool:
        """Return true if there is a Node with data.
        
        Returns:
            bool: True if data is in this list, False otherwise"""
        if self.head == None:
            return False
        current: Node = self.head
        while current and current.data is not data:
            current = current.next
        if current and current.data == data:
            return True
        return False



@dataclass
class Graph:
    class Edges(TypedDict):
        incident: LinkedList
        outgoing: LinkedList

    edges: list[tuple[int, int]]
    nodes: list[int]
    adj_list: dict[int, Edges] = None
    V: int = 0
    E: int = 0

    def __post_init__(self):
        V = len(self.nodes)
        E = len(self.edges)
        self._adj_list()

    def _adj_list(self) -> None:
        self.adj_list = dict[int, self.Edges](
            (node, self.Edges(incident=LinkedList(), outgoing=LinkedList())) for node in self.nodes)
        for edge in self.edges:
            if(edge[0] not in self.adj_list[edge[1]]['incident']):
                self.adj_list[edge[1]]['incident'].append(edge[0])
            if(edge[1] not in self.adj_list[edge[0]]['outgoing']):
                self.adj_list[edge[0]]['outgoing'].append(edge[1])



my_tree = Graph(edges=e, nodes=n)

In [ ]:
from collections import deque

def bfs(G: Graph, starting_node: int) -> list[int]:
    """Given a graph with an already defined adjacency list and a starting node in the list
    returns a list of nodes in discovered order.
    
    Returns:
        - List of int representing nodes"""
    
    discovered = list()
    q = deque[int]()
    v = set[int]()
    q.append(starting_node)
    v.add(q[0])
    # print(adj_list[5].head)
    while len(q) > 0:
        c_node = q[0]
        list_head = G.adj_list[c_node]['outgoing'].head
        while list_head is not None:
            if (list_head.data not in v):
                v.add(list_head.data)
                q.append(list_head.data)
            list_head = list_head.next
        discovered.append(c_node)
        q.popleft()
    return discovered

assert(bfs(my_tree, 1)==[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])
assert(bfs(my_tree, 2)==[2, 4, 5, 6, 1, 8, 9, 10, 11, 12, 3, 7, 13, 14])
assert(bfs(my_tree, 15)==[15, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14])

[2, 4, 5, 6, 1, 8, 9, 10, 11, 12, 3, 7, 13, 14]


In [ ]:
def dfs(G: Graph, starting_node: int) -> list[int]:
    """Given a graph with an already defined adjacency list and a starting node in the list
    returns a list of nodes in discovered order.
    
    Returns:
        - List of int representing nodes"""
    
    found = list()
    q = deque[int]()
    explored = set[int]()
    parent = dict[int,int]()

    q.append(starting_node)
    parent[q[0]] = None
    while len(q) > 0:
        c_node = q[-1]
        if c_node not in explored:
            explored.add(c_node)
            if(parent[c_node]):
                found.insert(0, c_node)
            list_head = G.adj_list[c_node]['outgoing'].head
            while list_head is not None:
                q.append(list_head.data)
                parent[list_head.data] = c_node
                list_head = list_head.next
        else:
            q.pop()
    found.append(starting_node)
    return found

assert(dfs(my_tree, 1)==[10, 8, 9, 4, 2, 5, 11, 12, 6, 13, 14, 7, 3, 1])
assert(dfs(my_tree, 15)==[10, 8, 9, 4, 2, 5, 11, 12, 6, 13, 14, 7, 3, 1, 15])
assert(dfs(my_tree, 6)==[12, 13, 14, 7, 8, 9, 4, 10, 11, 5, 2, 1, 3, 6])

[10, 8, 9, 4, 2, 5, 11, 12, 6, 13, 14, 7, 3, 1, 15]
[12, 13, 14, 7, 8, 9, 4, 10, 11, 5, 2, 1, 3, 6]


## Strong connectivity
"...a directed graph is strongly connected if, for every two nodes u and
v, there is a path from u to v and a path from v to u." (pg. 98)

"(3.16) If u and v are mutually reachable, and v and w are mutually reachable,
then u and w are mutually reachable." (pg. 98) which is pretty much a transitive property of connectivity.

A linear time algorithm for finding out if a directed grap is strongly connected involves running bfs on s for G and G<sup>rev</sup>. if there are nodes that are in one of the resulting set (or tree) but not the other, then the graph is not strongly connected. (pg. 98) Even if the graph is not strongly connected, the matching nodes of bfs(s) on G and G<sup>rev</sup> describe a strong component, or the strong component containing s. (pg. 99)

"(3.17) For any two nodes s and t in a directed graph, their strong components
are either identical or disjoint." (pg. 98)